# Protocolo do artigo — split por paciente

Partição **única e congelada** usada por todos os experimentos do artigo
(`plano_experimental_artigo_atualizado_2026-10-22.md`, seção 2.1):

| Partição | Pacientes | Uso |
|---|---:|---|
| `dev` (5 folds) | 70% | ajuste de pesos; folds do CV |
| `val` | 10% | tuning, seleção de checkpoint |
| `test` (INCA) | 20% | só na avaliação final (Etapa C) |

As porcentagens são de **pacientes**, não de frames. A divisão é **estratificada** por tamanho do
paciente: 15 dos 114 pacientes têm de 3 a 12 vídeos e concentram 564 dos 961 frames anotados. Sem
estratificar, a validação de 11 pacientes poderia ter de 40 a 207 frames.

A regra está em `src/data/splits.py:make_protocol_split`. O arquivo versionado é
`configs/splits/inca-protocolo-s42.csv`, gerado por `scripts/make_protocol_split.py`, que se recusa a
sobrescrever sem `--force`. Este notebook só **audita** o arquivo: não gera nada.

## Como a divisão acontece e por quê

### Passo a passo

A unidade de divisão é o **paciente**. Todos os frames e vídeos de um paciente caem juntos na mesma
partição (e, no `dev`, no mesmo fold). A regra, em ordem:

1. **Estratos.** Cada paciente é marcado como `multi_video` (3 a 12 vídeos; são 15) ou `single_video`
   (um vídeo de ~4 frames; são 99).
2. **Cotas em pacientes.** Dos 114 pacientes: `test` = 20% → 23, `val` = 10% → 11, `dev` = o resto → 80
   (arredondamento *half-up*, para não depender do arredondamento do banqueiro do `round()`).
   O estrato `multi_video` recebe a cota proporcional (3 test, 2 val, 10 dev) e o `single_video` completa
   o total (20 test, 9 val, 70 dev).
3. **Sorteio dentro de cada estrato.** Os pacientes, ordenados por id, são embaralhados com
   `RandomState(42)` e fatiados na ordem: primeiros → `test`, seguintes → `val`, restantes → `dev`.
4. **Folds do `dev` (5 folds de 16 pacientes).**
   - `multi_video`: ordena por nº de frames (decrescente) e distribui em **serpentina**
     (0,1,2,3,4,4,3,2,1,0). Cada fold recebe exatamente 2 pacientes pesados, um grande e um pequeno.
   - `single_video`: cada paciente vai para o fold com **menos pacientes** até então, o que mantém os folds
     com diferença máxima de 1 paciente.
5. **Congelamento.** O resultado é gravado em `configs/splits/inca-protocolo-s42.csv` (+ JSON com seed,
   sha256 da tabela de frames e git sha). Nenhum experimento recalcula a divisão: todos leem o arquivo.

Em cada execução, o arquivo vira o campo `split` dos frames: `train` = pacientes `dev` dos folds de
treino; `val` = a partição `val`; `test` = o fold `dev` avaliado no CV. O `test` INCA fica fora de
tudo (NaN) até a avaliação final.

### Por que assim

| Decisão | Motivo |
|---|---|
| **Por paciente, não por frame ou vídeo** | Frames do mesmo vídeo e vídeos do mesmo paciente são quase réplicas (mesma anatomia, mesmo equipamento). Dividir por frame deixa o modelo "ver" o paciente no treino e ser medido nele no teste: vazamento que infla o Dice. |
| **Proporções em pacientes** | O split antigo (`split_by_group`) mirava proporção de *frames*; com seed 42 a "validação de 10%" saía com 25 pacientes e o "teste de 20%" com 20. Contar pacientes torna 70/10/20 literal. |
| **Estratificado por nº de vídeos** | 15 pacientes concentram 564 de 961 frames (59%). Sem estratificar, a validação de 11 pacientes variaria de 40 a 207 frames conforme a seed, e um fold podia ficar sem nenhum paciente pesado. Estratificar fixa quantos pesados entram em cada partição/fold. |
| **5 folds no `dev`** | Com ~80 pacientes, um único hold-out tem variância alta. O CV dá média ± desvio entre folds para comparar supervisionado × SSL. |
| **`val` separado dos folds** | Tuning de hiperparâmetros e seleção de checkpoint usam só o `val`; assim o fold avaliado no CV não influencia essas escolhas. |
| **`test` intocado até a Etapa C** | Evita escolher método olhando o teste. |
| **Arquivo versionado + auditoria** | Garante que todos os métodos veem exatamente os mesmos pacientes; este notebook refaz a regra e confere igualdade com o CSV. |

### Isso está de acordo com a literatura (MICCAI etc.)?

**Sim no essencial, com ressalvas que vale declarar no artigo.**

O que é prática padrão e está coberto:
- **Split no nível do paciente** é o requisito mínimo; vazamento entre treino e teste é a falha
  metodológica mais citada em revisões de IA em imagem médica (p.ex. Varoquaux & Cheplygina, 2022;
  Kapoor & Narayanan, 2023). Revisores de MICCAI/MIDL checam isso.
- **Teste held-out usado uma vez**, com tuning em partição separada.
- **Split fixo, versionado e reprodutível**, o mesmo para todos os métodos comparados.
- **Média ± desvio entre folds**, em vez de um único número.

O que *não* é convenção, e sim escolha nossa:
- **Estratificar por nº de vídeos.** A literatura costuma estratificar por classe/rótulo, centro ou
  scanner. Aqui o desbalanceamento relevante é a distribuição de frames por paciente, então é uma
  adaptação justificada pelos dados, não uma regra publicada. Deve ser descrita (e justificada) no
  texto.
- **Seed única.** Um único sorteio de partição é comum, mas a conclusão pode depender dele. Se o
  orçamento permitir, repetir o protocolo com 2–3 seeds de split como análise de sensibilidade.

Limitações que o split não resolve:
- O **teste tem 23 pacientes (239 frames; só 3 multi-vídeo)**. É pequeno: intervalos de confiança são
  largos. Reportar IC por *bootstrap de pacientes* (reamostrar pacientes, não frames) e não afirmar
  diferenças menores que o IC.
- **Um único centro (INCA).** O teste é interno, então não mede generalização entre instituições;
  declarar isso como limitação. O UNM, se usado, só como teste externo, nunca no tuning.
- Vídeos do mesmo paciente têm correlação alta: testes estatísticos devem ter o **paciente** como unidade.

**É a melhor forma possível?** Entre as opções razoáveis, é uma das mais defensáveis para este
dataset. Alternativas seriam: (a) CV aninhado, mais rigoroso porém caro e sem ganho claro com o `val`
fixo; (b) CV com repetição de seeds, que reduz a dependência de um sorteio, mas multiplica o custo
computacional. Importa pouco *qual* esquema exato, desde que seja por paciente, sem vazamento e
reportado com transparência; o que reprova artigo é vazamento, não a escolha entre 5 folds e 10.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../")

import pandas as pd

from src.data.splits import load_protocol_split, make_protocol_split, summarize_protocol_split
from src.data.vfss_frame_dataset import load_video_frame_dataframe

DATASET_PATH = "/data_ssd/caioseda/data/dataset_inca_all_frames/"
SPLIT_FILE = "configs/splits/inca-protocolo-s42.csv"

frames = load_video_frame_dataframe(DATASET_PATH)
split = load_protocol_split(SPLIT_FILE)
split.head()

,paciente_id,partition,fold,stratum,n_videos,n_frames
0,1,test,<NA>,multi_video,12,60
1,2,dev,3,multi_video,8,33
2,3,dev,2,multi_video,6,30
3,4,dev,1,multi_video,4,20
4,5,dev,0,multi_video,3,15


## Contagens por partição e por fold

In [3]:
summary = summarize_protocol_split(split)
summary["% pacientes"] = (100 * summary.patients / len(split)).round(1)
summary["% frames"] = (100 * summary.frames / len(frames)).round(1)
summary

,partition,fold,patients,multi_video_patients,videos,frames,% pacientes,% frames
0,dev,<NA>,80,10,150,612,70.2,63.7
1,val,<NA>,11,2,26,110,9.6,11.4
2,test,<NA>,23,3,56,239,20.2,24.9
3,dev,0,16,2,27,121,14.0,12.6
4,dev,1,16,2,29,117,14.0,12.2
5,dev,2,16,2,30,127,14.0,13.2
6,dev,3,16,2,31,124,14.0,12.9
7,dev,4,16,2,33,123,14.0,12.8


## Auditoria

1. O arquivo versionado é exatamente o que a regra produz (ninguém o editou à mão).
2. Todo paciente da tabela de frames está em exatamente uma partição.
3. Só pacientes de `dev` têm fold.

In [4]:
regenerated = make_protocol_split(frames, seed=42)
regenerated["paciente_id"] = regenerated.paciente_id.astype(str)
pd.testing.assert_frame_equal(regenerated.reset_index(drop=True), split, check_dtype=False)

assert set(frames.paciente_id.astype(str)) == set(split.paciente_id)
assert not split.paciente_id.duplicated().any()
assert split[split.partition == "dev"].fold.notna().all()
assert split[split.partition != "dev"].fold.isna().all()
print("split congelado == regra; pacientes disjuntos; folds só no dev")

split congelado == regra; pacientes disjuntos; folds só no dev


## Pacientes multi-vídeo por partição e fold

In [5]:
(split[split.stratum == "multi_video"]
 .sort_values(["partition", "fold", "n_frames"], ascending=[True, True, False])
 [["paciente_id", "partition", "fold", "n_videos", "n_frames"]])

,paciente_id,partition,fold,n_videos,n_frames
6,7,dev,0,10,50
4,5,dev,0,3,15
14,15,dev,1,11,43
3,4,dev,1,4,20
7,8,dev,2,10,41
2,3,dev,2,6,30
8,9,dev,3,9,35
1,2,dev,3,8,33
12,13,dev,4,10,35
10,11,dev,4,9,34
